In [3]:
import pandas as pd
import numpy as np

def answer_one():
    # Load Energy data
    energy = pd.read_excel(
        'Energy Indicators.xlsx',
        skiprows=17,
        skipfooter=38,
        usecols=[2, 3, 4, 5],
        names=['Country', 'Energy Supply', 'Energy Supply per Capita', '% Renewable'],
        engine='openpyxl'
    )
    
    # Replace "..." with np.NaN
    energy.replace("...", np.nan, inplace=True)
    
    # Convert Energy Supply to gigajoules
    energy['Energy Supply'] *= 1000000
    
    # Clean country names
    energy['Country'] = energy['Country'].str.replace(r'\d+', '', regex=True)
    energy['Country'] = energy['Country'].str.replace(r'\s*\(.*\)', '', regex=True)
    energy.replace({
        'Republic of Korea': 'South Korea',
        'United States of America': 'United States',
        'United Kingdom of Great Britain and Northern Ireland': 'United Kingdom',
        'China, Hong Kong Special Administrative Region': 'Hong Kong'
    }, inplace=True)

    # Load GDP data
    gdp = pd.read_csv('API_NY.GDP.MKTP.CD_DS2_en_csv_v2_26433.csv', skiprows=4)
    gdp.rename(columns={'Country Name': 'Country'}, inplace=True)
    gdp.replace({
        'Korea, Rep.': 'South Korea',
        'Iran, Islamic Rep.': 'Iran',
        'Hong Kong SAR, China': 'Hong Kong'
    }, inplace=True)
    gdp = gdp[['Country'] + [str(year) for year in range(2006, 2016)]]

    # Load ScimEn data
    scimen = pd.read_excel('scimagojr country rank 1996-2024.xlsx', engine='openpyxl')
    scimen_top15 = scimen[scimen['Rank'] <= 15]

    # Merge datasets
    merged_df = pd.merge(scimen_top15, energy, on='Country')
    merged_df = pd.merge(merged_df, gdp, on='Country')
    merged_df.set_index('Country', inplace=True)

    # Return only the required columns (excluding any extra like 'Region')
    columns = ['Rank', 'Documents', 'Citable documents', 'Citations', 'Self-citations',
               'Citations per document', 'H index', 'Energy Supply',
               'Energy Supply per Capita', '% Renewable'] + [str(year) for year in range(2006, 2016)]
    
    return merged_df[columns]


In [4]:
df = answer_one()
print(df)

                    Rank  Documents  Citable documents  Citations  \
Country                                                             
China                  1     472465             470142    6591474   
United States          2     222772             217929    4131736   
India                  3      96975              94714    1243636   
United Kingdom         4      61946              60287    1326766   
Japan                  5      61939              61307     813472   
Germany                6      55466              54296     932195   
Russian Federation     7      49938              49562     268391   
Canada                 8      44877              44004    1082361   
Italy                  9      42635              40695     767280   
South Korea           10      42534              42174     761076   
Iran                  11      35889              35484     798505   
France                12      33407              32698     637552   
Spain                 13      3252

C:\Users\Admin\AppData\Local\Temp\ipykernel_18448\2234204488.py:16: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  energy.replace("...", np.nan, inplace=True)


In [5]:
def answer_two():
    Top15 = answer_one()
    avgGDP = Top15[[str(year) for year in range(2006, 2016)]].mean(axis=1).sort_values(ascending=False)
    return avgGDP
print(answer_two())

Country
United States         1.572243e+13
China                 6.927707e+12
Japan                 5.239642e+12
Germany               3.590729e+12
United Kingdom        2.777505e+12
France                2.692000e+12
Italy                 2.152983e+12
Brazil                1.988889e+12
Russian Federation    1.666746e+12
Canada                1.616359e+12
India                 1.602352e+12
Spain                 1.406644e+12
South Korea           1.221328e+12
Australia             1.207997e+12
Iran                  4.567516e+11
dtype: float64


C:\Users\Admin\AppData\Local\Temp\ipykernel_18448\2234204488.py:16: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  energy.replace("...", np.nan, inplace=True)


In [6]:
def answer_three():
    Top15 = answer_one()
    avgGDP = answer_two()
    sixth_country = avgGDP.index[5]  # 6th largest avg GDP
    gdp_change = Top15.loc[sixth_country, '2015'] - Top15.loc[sixth_country, '2006']
    return gdp_change
print(answer_three())

C:\Users\Admin\AppData\Local\Temp\ipykernel_18448\2234204488.py:16: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  energy.replace("...", np.nan, inplace=True)
C:\Users\Admin\AppData\Local\Temp\ipykernel_18448\2234204488.py:16: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  energy.replace("...", np.nan, inplace=True)


124621907951.68018


In [7]:
def answer_four():
    Top15 = answer_one()
    Top15['Self-citation ratio'] = Top15['Self-citations'] / Top15['Citations']
    max_country = Top15['Self-citation ratio'].idxmax()
    max_value = Top15.loc[max_country, 'Self-citation ratio']
    return (max_country, max_value)
print(answer_four())

('China', np.float64(0.6970630544852335))


C:\Users\Admin\AppData\Local\Temp\ipykernel_18448\2234204488.py:16: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  energy.replace("...", np.nan, inplace=True)


In [8]:
def answer_five():
    Top15 = answer_one()
    Top15['Estimated Population'] = Top15['Energy Supply'] / Top15['Energy Supply per Capita']
    third_most_populous = Top15['Estimated Population'].sort_values(ascending=False).index[2]
    return third_most_populous
print(answer_five())

United States


C:\Users\Admin\AppData\Local\Temp\ipykernel_18448\2234204488.py:16: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  energy.replace("...", np.nan, inplace=True)


In [9]:
def answer_six():
    Top15 = answer_one()
    Top15['Estimated Population'] = Top15['Energy Supply'] / Top15['Energy Supply per Capita']
    Top15['Citable docs per Capita'] = Top15['Citable documents'] / Top15['Estimated Population']
    correlation = Top15['Citable docs per Capita'].corr(Top15['Energy Supply per Capita'])
    return correlation
print(answer_six())

0.6905473831164103


C:\Users\Admin\AppData\Local\Temp\ipykernel_18448\2234204488.py:16: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  energy.replace("...", np.nan, inplace=True)


In [10]:
def answer_seven():
    Top15 = answer_one()
    ContinentDict = {
        'China': 'Asia', 'United States': 'North America', 'Japan': 'Asia',
        'United Kingdom': 'Europe', 'Russian Federation': 'Europe', 'Canada': 'North America',
        'Germany': 'Europe', 'India': 'Asia', 'France': 'Europe',
        'South Korea': 'Asia', 'Italy': 'Europe', 'Spain': 'Europe',
        'Iran': 'Asia', 'Australia': 'Australia', 'Brazil': 'South America'
    }
    
    Top15['Continent'] = Top15.index.to_series().map(ContinentDict)
    Top15['Estimated Population'] = Top15['Energy Supply'] / Top15['Energy Supply per Capita']
    
    result = Top15.groupby('Continent')['Estimated Population'].agg(['size', 'sum', 'mean', 'std'])
    return result
print(answer_seven())

               size           sum          mean           std
Continent                                                    
Asia              5  2.898666e+09  5.797333e+08  6.790979e+08
Australia         1  2.331602e+07  2.331602e+07           NaN
Europe            6  4.579297e+08  7.632161e+07  3.464767e+07
North America     2  3.528552e+08  1.764276e+08  1.996696e+08
South America     1  2.059153e+08  2.059153e+08           NaN


C:\Users\Admin\AppData\Local\Temp\ipykernel_18448\2234204488.py:16: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  energy.replace("...", np.nan, inplace=True)
